# l3d-bench: one study run (M1-M3)

**Before running** (once per notebook)
- Accelerator: **GPU T4 x2** (the code uses one GPU; keep the same type for every run, the run card records it)
- Internet: **on**
- Add data: `khoatrytopublish/l3d-train`, `l3d-val`, `l3d-test`
- Add-ons > Secrets: tick `RCLONE_CONFIG_GDRIVE`

**How to use it:** set `RUN_ID` below to your row of `runs/manifest.csv`, then Save Version > Save & Run All.
If the log ends with `paused` (Kaggle's 12-hour limit), run the same notebook again: it continues from the checkpoint on Drive.
When it ends with `done`, mark the row done in the manifest. Running a finished run again does nothing.

In [ ]:
RUN_ID = "m1__plain__official__s0"   # <method>__<variant>__<split>__s<seed>
PILOT = False                        # True: 2 epochs on 8 frames, saved under _pilot__<RUN_ID>
OVERFIT_CHECK = False                # True: gate G3 overfit test first (10 frames, 300 steps, ~10 min)
COMMIT = "main"                      # pin the study-v1 commit for study runs

In [ ]:
import os, subprocess
REPO, WORK = "https://github.com/KhoaLeQuangMinh/l3d-bench.git", "/kaggle/working/l3d-bench"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "-q", REPO, WORK], check=True)
subprocess.run(["git", "-C", WORK, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", WORK, "checkout", "-q", COMMIT], check=True)
subprocess.run(["git", "-C", WORK, "pull", "-q", "--ff-only"], check=False)   # no-op on a pinned hash
os.chdir(WORK)
subprocess.run("pip install -q medpy && apt-get -qq update && apt-get -qq install -y rclone > /dev/null", shell=True, check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"

In [ ]:
if OVERFIT_CHECK:
    !python scripts/overfit_check.py

In [ ]:
!python -m l3d_bench.run --run-id {RUN_ID} {"--pilot" if PILOT else ""}